# 04 - r6 top-k sweep (gated on the fetch)

**Gate:** requires `DONE_fetch.flag` on Drive. CPU-heavy boosting, no GPU
math - runs fine on the A100 runtime's CPUs after 03, or on a separate CPU
runtime.

Stage 1 is the k=10 calibration against the frozen tables. **Standing rule:
if label kappa < 0.8 in any discipline, STOP at that cell and surface the
numbers - the k sweep would be confounded by snapshot drift and the drift
measurement itself becomes the exhibit.**

In [ ]:
import sys
sys.path.insert(0, "/content/drive/MyDrive/who-inherits")  # for colab_common if bundle not yet unzipped
try:
    import colab_common as cc
except ImportError:
    # colab_common ships inside the bundle; bootstrap: mount, unzip, import
    from google.colab import drive as _d; _d.mount("/content/drive")
    import subprocess
    subprocess.run(["unzip", "-q", "-o",
                    "/content/drive/MyDrive/who-inherits/colab_bundle.zip",
                    "-d", "/content/work"], check=True)
    sys.path.insert(0, "/content/work/colab")
    import colab_common as cc
else:
    cc.mount_drive()
sys.path.insert(0, "/content/work/colab")
import colab_common as cc
cc.setup_workspace()
cc.verify_frozen_hashes()
print(cc.run_meta())

In [ ]:
assert (cc.DRIVE / "DONE_fetch.flag").exists(), \
    "fetch not finished - run 01_fetch_r5 first"
def field_env(field):
    return {"DATASET": field,
            "DATASET_PATH": f"data/clean_dataset_{field}.parquet"}

In [ ]:
ckpt = cc.start_checkpoint_thread()
kappas = {}
import json
for f in ["econ", "math", "physics", "neuro", "chemistry"]:
    cc.run_script("code/paper_pipeline/experiments/r6_topk_sweep.py",
                  env_extra=field_env(f))   # calibration + k grid + min-score
    cc.sync_to_drive()
    cal = json.load(open(f"/content/work/results/robustness/topk_partial/{f}_calibration_k10.json"))
    kappas[f] = cal["label_kappa"]
    print(f, "match:", cal["match_within_1e6"], "kappa:", cal["label_kappa"])
    assert cal["label_kappa"] >= 0.8, (
        f"KAPPA GATE FAILED for {f}: {cal['label_kappa']} < 0.8 - STOP. "
        "The k sweep is confounded by snapshot drift; report the calibration "
        "table itself (already synced to Drive) instead of any k-sweep claim.")
cc.stop_checkpoint_thread()
print("all kappas >= 0.8:", kappas)

In [ ]:
cc.run_script("code/paper_pipeline/experiments/r6_topk_sweep.py", args=("--merge",))
import json
s = json.load(open("/content/work/results/robustness/topk_sweep_summary.json"))
print("kappa gate:", s["kappa_gate_0.8"])
for f, blk in s["fields"].items():
    print(f, "branch stable across k:", blk["branch_stable_all_k"])

In [ ]:
cc.write_done_flag("topk")